# Week 8 Problem Set: Three Polls, Three Answers

**Instructions:** Complete all four tasks.

You have data from two polls of the same race that disagree. Your job: figure out why they disagree, reweight one of them, and write a recommendation.

---

**Lying with data, the checklist so far:**

1. **W1:** Conflating fixed and marginal costs.
2. **W2:** Presenting an observational comparison as a causal effect.
3. **W3:** Applying a result from one setting to a different one.
4. **W4:** Cherry-picking the winning arm from a multi-arm test.
5. **W5:** Treating an underpowered null as evidence of no effect.
6. **W7:** Reporting the complier comparison as a causal effect.
7. **W8:** Cherry-picking polls; house effects; ignoring nonresponse bias.

### Before you start

**Save your own copy first.** Go to **File → Save a copy in Drive**. A new tab opens with your own copy. Work in that tab; edits to the original are not saved.

**The data loads itself.** There is nothing to download or upload. The setup cell below pulls the data straight from the course repository; you just need to be online.

Stuck? See the Colab troubleshooting guide on the syllabus.

## Setup

In [ ]:
import pandas as pd
import numpy as np

phone = pd.read_csv('https://raw.githubusercontent.com/joshuakalla/data_science_campaigns_26/'
                    'main/weeks/wk08_trusting_polls/data/phone_poll.csv')
online = pd.read_csv('https://raw.githubusercontent.com/joshuakalla/data_science_campaigns_26/'
                     'main/weeks/wk08_trusting_polls/data/online_poll.csv')

# The population's true age distribution (from the voter file)
pop_age = {'18-29': 0.180, '30-44': 0.250, '45-64': 0.321, '65+': 0.249}

## Task 1: Unweighted toplines

The code below computes the unweighted topline for each poll. **Run both cells.**

In [ ]:
# Phone poll topline
phone_topline = phone['vote_intent'].value_counts(normalize=True).sort_index()
print('Phone poll (unweighted):')
print(phone_topline.round(3))

In [ ]:
# Online poll topline
online_topline = online['vote_intent'].value_counts(normalize=True).sort_index()
print('Online poll (unweighted):')
print(online_topline.round(3))

**Question 1:** The phone poll shows Rep +4 and the online poll shows Dem +8. In one sentence, why do they disagree?

*Your answer:*

## Task 1b: Is the 12-point gap just sampling noise?

The polls disagree by 12 points. Before blaming bias, check what random sampling alone could produce: the **margin of error** (the Week-5 confidence interval, now on a poll number). Run the cell.

In [ ]:
def poll_uncertainty(d, name):
    n = len(d)
    tl = d['vote_intent'].value_counts(normalize=True)
    p_dem, p_rep = tl.get('dem', 0), tl.get('rep', 0)
    lead = p_dem - p_rep
    se_lead = np.sqrt((p_dem + p_rep - lead**2)/n)
    lo, hi = lead - 1.96*se_lead, lead + 1.96*se_lead
    verdict = 'statistical TIE' if lo < 0 < hi else 'a real lead'
    print(f'{name}: lead {100*lead:+.1f} pp | each-number MoE ±{100*1.96*np.sqrt(0.25/n):.1f} pp '
          f'| lead 95% CI [{100*lo:+.1f}, {100*hi:+.1f}] -> {verdict}')

poll_uncertainty(phone,  'Phone poll ')
poll_uncertainty(online, 'Online poll')

**Question 1b** (several parts; take them one at a time):

(a) Each poll's margin of error on a single candidate is about ±3 points. Could sampling noise alone explain a *12-point* disagreement between the two polls?

(b) Is the phone poll's Rep +4 lead distinguishable from a tie?

(c) The margin of error shrinks if you poll more people, but the 12-point gap would barely move. In one or two sentences, what does that tell you about where the disagreement comes from?

(d) The online poll's Dem +8 *is* distinguishable from a tie. But look at the **lower end** of its 95% CI on the lead (printed above). What's the smallest Dem lead that poll is consistent with, and does that change how "safe" a Dem +8 headline feels?

*Your answers:*

## Task 2: Compare the samples

The code below compares the age distribution of each poll to the population. **Run it.**

In [ ]:
# Age distribution comparison
phone_age = phone['age_group'].value_counts(normalize=True).sort_index()
online_age = online['age_group'].value_counts(normalize=True).sort_index()
pop_age_series = pd.Series(pop_age).sort_index()

comparison = pd.DataFrame({
    'Population': pop_age_series,
    'Phone poll': phone_age,
    'Online poll': online_age
})
print(comparison.round(3))

**Before you move on:** If young voters (18-29) lean Democratic and old voters (65+) lean Republican, and the phone poll has far more 65+ respondents than the population actually has, which direction would you expect the phone poll’s topline to be biased? Would reweighting to fix the age imbalance move the topline toward or away from the Democrat?

Write your answer in 1–2 sentences.

**Your answer:**

*Replace this text with your answer.*

## Task 3: Reweight the phone poll

In the cells below, compute the reweighted Dem share for the phone poll. Follow the same steps from livecode:

1. Compute a weight for each age group: `pop_age[group] / phone_age[group]`.
2. Assign each respondent their group’s weight.
3. Compute the weighted Dem share: `(is_dem * weight).sum() / weight.sum()`.

The first cell gives you the weights. **Write the weighted mean yourself** in the second cell.

*Check: the reweighted Dem share should be higher than the unweighted Dem share (the phone poll was oversampling Republicans).*

In [ ]:
# Step 1-2: compute weights and assign to each respondent (pre-filled)
weights_dict = {}
for group in pop_age:
    weights_dict[group] = pop_age[group] / phone_age[group]

phone['weight'] = phone['age_group'].map(weights_dict)
phone['is_dem'] = (phone['vote_intent'] == 'dem').astype(int)
phone['is_rep'] = (phone['vote_intent'] == 'rep').astype(int)

print('Weights by age group:')
for g, w in weights_dict.items():
    print(f'  {g}: {w:.2f}')

In [ ]:
# Step 3: YOUR CODE HERE
# Compute the weighted Dem share and weighted Rep share.
# Formula: (column * weight).sum() / weight.sum()
# Print both, and the reweighted Dem lead.

**Question 3:** How much did reweighting move the phone poll’s topline? In one sentence, explain why the reweighted number is more trustworthy than the unweighted number.

*Your answer:*

**Before you move on:** Post-stratification (reweighting) corrects the age imbalance. But suppose that within the 65+ group, the people who answer phone polls are more Republican-leaning than the 65+ voters who don’t answer. Would reweighting by age fix this problem? Why or why not?

Write your answer in 2–3 sentences.

**Your answer:**

*Replace this text with your answer.*

## Task 4: Referee the two pollsters (≈250–300 words)

Two of your pollsters disagree by 12 points, and each has sent the campaign manager a pitch for why *their* number is the one to trust. The manager forwards you both and says: "They can't both be right! Which do I believe, and where does this race actually stand?"

**The phone pollster (Rep +4):**
> "Live-caller phone polling is the gold standard. We reach a real cross-section by random-digit dialing, our interviewers screen for likely voters, and live calls don't have the self-selection of opt-in web panels. The online poll's Dem +8 is a fantasy of whoever clicks a survey link: disproportionately young, online, and over-engaged. Our Rep +4 reflects the older, high-turnout electorate that actually votes in a midterm. Trust the mode with the track record."

**The online pollster (Dem +8):**
> "Phone response rates have collapsed below 5%. The few people who still pick up for a live caller are unrepresentative in ways no weight can see, and they skew old and Republican. Our panel is large, balanced on age, gender, region, and education, and weighted to the census. We reach the young and the busy that phones miss entirely. Rep +4 is a relic of a dying mode; Dem +8 is where the electorate actually is."

You are the campaign's polling analyst. Write your **ruling**: your adjudication for the manager, not a memo to either pollster. It must do all three:

**(a) The bias in each.** Each pollster is *right about the other* and *wrong about itself*. Name the specific flaw in each number using your Task 2 age composition (phone 41% age 65+; online 33% age 18–29). Where is each one's self-defense too generous?

**(b) Your reconciled estimate.** Use the Task 3 reweighting to give your single best read of the race. Say what reweighting fixes (the age imbalance) and what it cannot (nonresponse *within* age groups, the 2020 miss).

**(c) The ruling.** Which poll do you trust more, if either, and what do you tell the manager to plan around? Note whether these "leads" are even distinguishable from a statistical tie.

**Style:** Recommendation in the first sentence. \~250–300 words. At least one specific number.

**Ruling for the campaign manager**
**From:** You, Polling Analyst
**Re:** The phone-vs-online dispute

*Replace this text with your \~250–300 word ruling. Recommendation first.*

---

**Due at 4:00pm on Wednesday Nov 4**, to the **problem set** assignment on Canvas. Whatever you had at 5:55pm in class already went to the separate **in-class** assignment; that one is your attendance credit and you do not resubmit it.


## Before you submit

1. **Runtime → Restart session and run all.** Do this *after* you have finished every task and written your memo. It clears every variable and runs the notebook from top to bottom, in order, so the version you hand in is one that actually works start to finish.
2. **Check that every cell actually ran.** Scroll from the top. Every code cell should show a number in its left margin and its output below it. If the run stopped at a cell with an error, that is a cell you have not finished. Fix it, then restart and run all again.
3. **File → Print → Save as PDF.**
4. **Open the PDF and read it before you upload.** The PDF will look complete even when it isn't. Every heading and prompt prints whether or not the code ran. What matters is the **output**: under each code cell you should see a table, a number, or a plot. A red error box, or `In [ ]` with nothing beneath it, means that part did not run and will be graded as missing. Also check that your memo printed in full and that no plot is cut off at a page break.
5. Upload the PDF to Canvas.